# RFDC V4 -- Row Program, No-Ethernet Timing Bench

Purpose: isolate and *measure* the one thing that's actually new and unproven compared to
`RFDC_V4_Pump_Pi2_Probe_TTL_Loop_Qualification_Ethernet.ipynb` -- mid-run, Python-timed TX DMA
refills -- without any of the other new machinery (Ethernet, background thread, double
buffering) in the way.

**What changed vs `RFDC_V4_RowProgram_Design_FIXED_2_.ipynb`:**
- No Ethernet, no socket, no background sender thread. Nothing competes with the run loop
  for the GIL during a shot.
- No RX double buffering -- one buffer per RX lane, reused every shot, exactly like the
  qualification notebook. There's nothing to overlap without Ethernet, so it bought nothing
  here and only added PS DDR4 usage.
- **One `allocate()` per TX lane, not one per chunk.** The whole lane's waveform (all reps
  concatenated) is quantized once into a single PL buffer; each refill is a *slice* of that
  one buffer (`buf[c0:c1]`), not a separate allocation. This is the fix for "why 20 buffers" --
  chunking the DMA *transfer* doesn't require chunking the *allocation*.
- **Every refill is timestamped** against a model of when the row that consumes it actually
  starts (derived from the committed row table's own `gap_cycles`), so instead of guessing
  whether a 2 ms window is enough, this notebook prints the real margin -- or a negative
  number, in red, if a refill would have missed its deadline.
- A **standalone `transfer()` call-overhead benchmark runs first**, before anything is armed,
  so you have a real number for "how long does issuing a refill actually take on this board"
  instead of the placeholder guess (`TRANSFER_CALL_OVERHEAD_S_ESTIMATE`) that the earlier
  notebook only warned about.

**Status: not yet run against hardware.** Everything below follows from the same VHDL/register
map already validated in the qualification notebook; only the refill polling loop and the
single-buffer-with-slices pattern are new here.

## Cell 1 -- Overlay + imports (only this)

In [1]:
# ================= OVERLAY + IMPORTS (run this once) =================
import time
import numpy as np
import xrfclk
import xrfdc
from pynq import Overlay, allocate

BITFILE = "./final.bit"   # confirm this is the 256-row Pulse_Sequencer build
base = Overlay(BITFILE)
print("Overlay loaded:", BITFILE)


Overlay loaded: ./final.bit


## Cell 2 -- User configuration

In [2]:
# ================= USER CONFIG =================
AXIS_BEAT_HZ = 15.36e6
SAMPLES_PER_BEAT = 8
SEQ_CLK_HZ = 99_999_985.0
FS_HZ = 122.88e6
AMPLITUDE = 32760

DAC_A_NCO_MHZ = 10.0
DAC_B_NCO_MHZ = 0.2

DMA_MAX_BYTES = (1 << 26) - 1
CAP_MAX_S = 0.260
TX_MAX_S = 0.136

LOOP_COUNT = 5           # keep small for a bench run; bump once margins look safe
DEBUG_VERBOSE = True

EXPECTED_IP_PATHS = {
    "sequencer": "radio/AXI_Pulse_Sequencer_0",
    "tx_gate_b": "radio/AXI_TX_Multi_Gate_0",
    "tx_gate_a": "radio/AXI_TX_Multi_Gate_1",
    "cap_gate_b": "radio/receiver/channel_20/AXI_Capture_Gate_0",
    "cap_gate_a": "radio/receiver/channel_21/AXI_Capture_Gate_0",
    "rx_dma_b": "radio/receiver/channel_20/axi_dma_real",
    "rx_dma_a": "radio/receiver/channel_21/axi_dma_real",
    "tx_dma_b": "radio/axi_dma_dac_0",
    "tx_dma_a": "radio/axi_dma_dac_1",
}
print("Config loaded.")


Config loaded.


## Cell 3 -- IP resolution

In [3]:
# ================= IP RESOLUTION =================
def get_by_path(root, path):
    obj = root
    for part in path.split('/'):
        obj = getattr(obj, part)
    return obj

missing = [p for p in EXPECTED_IP_PATHS.values() if p not in base.ip_dict]
if missing:
    print("Missing expected HWH paths:")
    for p in missing:
        print("  ", p)
    raise KeyError("The loaded .hwh does not match the validated A/B topology.")

seq   = get_by_path(base, EXPECTED_IP_PATHS['sequencer'])
tx_b  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_b'])
tx_a  = get_by_path(base, EXPECTED_IP_PATHS['tx_gate_a'])
cap_b = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_b'])
cap_a = get_by_path(base, EXPECTED_IP_PATHS['cap_gate_a'])
dma_rb = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_b'])
dma_ra = get_by_path(base, EXPECTED_IP_PATHS['rx_dma_a'])
dma_tb = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_b'])
dma_ta = get_by_path(base, EXPECTED_IP_PATHS['tx_dma_a'])

rfdc = get_by_path(base, 'radio/rfdc')
dac_b = rfdc.dac_tiles[0].blocks[0]
dac_a = rfdc.dac_tiles[2].blocks[0]
adc_b = rfdc.adc_tiles[2].blocks[0]
adc_a = rfdc.adc_tiles[2].blocks[1]
print('IP resolved.')


IP resolved.


## Cell 4 -- Register map + unit-conversion helpers

In [4]:
# ================= REGISTER MAP =================
SEQ_ROW_SEL=0x00; SEQ_MASK=0x04; SEQ_GAP=0x08
SEQ_DUR0=0x0C; SEQ_DUR1=0x10; SEQ_DUR2=0x14; SEQ_DUR3=0x18
SEQ_COMMIT=0x1C; SEQ_TABLE_LEN=0x20; SEQ_ENABLE=0x24
SEQ_RESET=0x28; SEQ_ROW_PTR=0x2C; SEQ_CDC_OVERRUN=0x30
SEQ_TTL_ARM=0x34; SEQ_TTL_MODE=0x38; SEQ_TTL_STATUS=0x3C
SEQ_TTL_STATUS_CLEAR=0x40; SEQ_TTL_EDGE_COUNT=0x44
TTL_ARMED=1<<0; TTL_RUNNING=1<<1; TTL_RUN_DONE=1<<2

TXM_SEG_SEL=0x00; TXM_ACTIVE_LEN=0x04; TXM_GAP_LEN=0x08; TXM_COMMIT=0x0C
TXM_NUM_SEGMENTS=0x10; TXM_SW_START=0x14; TXM_SEG_PTR=0x18; TXM_STATUS=0x1C
# NOTE: TXM_SW_START (0x14) and TXM_SEG_PTR (0x18) exist in AXI_TX_Multi_Gate.vhd's
# register map but were missing here. SW_START independently forces the gate open
# using INTERNAL-table mode, regardless of the Pulse_Sequencer/hw_start path -- this
# is the only way to open a gate for a bench/debug test without a committed sequencer
# program. TX_OVERRUN here is actually STATUS bit1 = CONTROL_OVERRUN per the VHDL
# comment (COMMIT/SW_START issued too close together), not a data overrun flag; kept
# the existing name so the rest of the notebook does not need touching.
TX_BUSY=1<<0; TX_OVERRUN=1<<1

CAP_LENGTH=0x00; CAP_STATUS=0x08; CAP_CLEAR=0x0C
CAP_BUSY=1<<0; CAP_OVERFLOW=1<<1

DMASR_HALTED=1<<0; DMASR_IDLE=1<<1; DMASR_ERR_MASK=(1<<4)|(1<<5)|(1<<6)

LANE_TX_B, LANE_TX_A, LANE_RX_B, LANE_RX_A = 0, 1, 2, 3
LANE_BIT = {"TX_B": LANE_TX_B, "TX_A": LANE_TX_A, "RX_B": LANE_RX_B, "RX_A": LANE_RX_A}
TX_LANES = ("TX_A", "TX_B")
RX_LANES = ("RX_A", "RX_B")

def beats_for(seconds):
    return max(1, int(round(seconds * AXIS_BEAT_HZ)))

def samples_for(beats):
    return int(beats) * SAMPLES_PER_BEAT

def seq_cycles_for(seconds):
    return max(1, int(round(seconds * SEQ_CLK_HZ)))

def pack_iq(i, q):
    if len(i) != len(q):
        raise ValueError('I/Q length mismatch')
    out = np.empty(2*len(i), dtype=np.int16)
    out[0::2] = i
    out[1::2] = q
    return out
print('Register map ready.')


Register map ready.


## Cell 5 -- Waveform primitives

Unchanged from the row-program design: `chirp()` (sine is `f0==f1`), `zeros()`,
`envelope()`, `waveform(*parts)`, `quantize_iq()` (final step only), `chirp_train()`.

In [5]:
# ================= WAVEFORM PRIMITIVES =================
def chirp(f0_mhz, f1_mhz, duration_s, nco_mhz):
    n = samples_for(beats_for(duration_s))
    t = np.arange(n, dtype=np.float64) / FS_HZ
    f0 = (f0_mhz - nco_mhz) * 1e6
    f1 = (f1_mhz - nco_mhz) * 1e6
    k = (f1 - f0) / duration_s if duration_s > 0 else 0.0
    ph = 2*np.pi*(f0*t + 0.5*k*t*t)
    return np.exp(1j*ph).astype(np.complex128)

def zeros(duration_s):
    n = samples_for(beats_for(duration_s))
    return np.zeros(n, dtype=np.complex128)

def envelope(arr, kind="gaussian", **params):
    n = len(arr)
    if kind == "gaussian":
        sigma = params.get("sigma", 0.25) * n
        x = np.arange(n) - (n-1)/2.0
        win = np.exp(-0.5*(x/sigma)**2)
    else:
        raise ValueError(f"unknown envelope kind: {kind}")
    return arr * win

def waveform(*parts):
    return np.concatenate(parts) if parts else np.zeros(0, dtype=np.complex128)

def quantize_iq(arr):
    i = np.round(AMPLITUDE * arr.real).astype(np.int16)
    q = np.round(AMPLITUDE * arr.imag).astype(np.int16)
    return pack_iq(i, q)

def chirp_train(freqs_mhz, pulse_s, gap_s, nco_mhz):
    parts = []
    for idx, f in enumerate(freqs_mhz):
        parts.append(chirp(f, f, pulse_s, nco_mhz))
        if idx < len(freqs_mhz) - 1:
            parts.append(zeros(gap_s))
    return waveform(*parts)

print('Waveform primitives ready.')


Waveform primitives ready.


## Cell 6 -- Row helpers: `repeat()` and `print_program()`

In [6]:
# ================= ROW HELPERS =================
def repeat(rows, n):
    return list(rows) * n

def _row_lanes(row):
    lanes = []
    for lane in TX_LANES:
        if lane in row.get("tx", {}):
            lanes.append(lane)
    for lane in RX_LANES:
        if lane in row.get("rx", {}):
            lanes.append(lane)
    return lanes

def print_program(rows):
    print(f"{'row':>4}  {'gap_s':>10}  {'lanes (dur_s)':<50} {'refill'}")
    for idx, row in enumerate(rows):
        parts = []
        for lane in _row_lanes(row):
            if lane in TX_LANES:
                arr = row['tx'][lane]
                dur_s = len(arr) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ
                parts.append(f"{lane}:{dur_s*1e3:.4f}ms({len(arr)}smp)")
            else:
                dur_s = row['rx'][lane]
                parts.append(f"{lane}:{dur_s*1e3:.4f}ms")
        refill = ','.join(row.get('refill', [])) or '-'
        print(f"{idx:>4}  {row['gap_s']*1e3:>9.4f}ms  {', '.join(parts):<50} {refill}")

print('repeat() and print_program() ready.')


repeat() and print_program() ready.


## Cell 7 -- The program (same sequence as the design that hung, unchanged on purpose)

Kept identical to `RFDC_V4_RowProgram_Design_FIXED_2_.ipynb`'s state-prep / spectroscopy /
detection example so this bench is testing the same timing that failed, not an easier case.

In [7]:
# ================= PROGRAM DEFINITION =================
ROW_MARGIN_S = 5e-6

def gap_after(*durations_s):
    return max(durations_s) + ROW_MARGIN_S


state_prep_dacA = waveform(
    chirp(15, 15, 1e-3, DAC_A_NCO_MHZ),
    chirp(5, 10, 1e-3, DAC_A_NCO_MHZ),
)

state_prep_dacB = chirp(
    5, 25, 20e-3, DAC_B_NCO_MHZ
)

STATE_PREP_REPS = 20


# Chunk 0 of TX_B is armed before TTL_ARM.
# Therefore the FIRST TX_A interval does not refill anything.
state_prep_rows = [
    {
        "gap_s": gap_after(2e-3),
        "tx": {"TX_A": state_prep_dacA},
    },
    {
        "gap_s": gap_after(20e-3),
        "tx": {"TX_B": state_prep_dacB},
    },
]


# After each completed TX_B pulse, the following TX_A interval is
# the opportunity to load the NEXT TX_B chunk.
for rep in range(1, STATE_PREP_REPS):
    state_prep_rows += [
        {
            "gap_s": gap_after(2e-3),
            "tx": {"TX_A": state_prep_dacA},
            "refill": ["TX_B"],
        },
        {
            "gap_s": gap_after(20e-3),
            "tx": {"TX_B": state_prep_dacB},
        },
    ]


SPECTROSCOPY_S = 5e-3

spectroscopy_rows = [
    {
        "gap_s": gap_after(SPECTROSCOPY_S),
        "tx": {},
        "rx": {},
    },
]


probe = chirp_train(
    [190, 191, 192, 193, 194],
    pulse_s=10e-6,
    gap_s=1e-6,
    nco_mhz=DAC_A_NCO_MHZ,
)

CAPTURE_S = len(probe) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ

detection_rows = [
    {
        "gap_s": gap_after(CAPTURE_S),
        "tx": {"TX_A": probe},
        "rx": {
            "RX_A": CAPTURE_S,
            "RX_B": CAPTURE_S,
        },
    },
]


rows = (
    state_prep_rows
    + spectroscopy_rows
    + detection_rows
)

print_program(rows)


 row       gap_s  lanes (dur_s)                                      refill
   0     2.0050ms  TX_A:2.0000ms(245760smp)                           -
   1    20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   2     2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   3    20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   4     2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   5    20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   6     2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   7    20.0050ms  TX_B:20.0000ms(2457600smp)                         -
   8     2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
   9    20.0050ms  TX_B:20.0000ms(2457600smp)                         -
  10     2.0050ms  TX_A:2.0000ms(245760smp)                           TX_B
  11    20.0050ms  TX_B:20.0000ms(2457600smp)                         -
  12     2.0050ms  TX_A:2.0000ms(245760smp)  

## Cell 8 -- Compile: derive mask/dur, ONE buffer per lane, refill deadlines

Same mask/DUR derivation as before. New: `row_start_s[idx]` models the wall-clock time (from
`TTL_ARM`) each row *starts*, purely from the committed `gap_cycles` -- this is what lets the
run loop print a real number for "how much margin did this refill actually have" instead of
just hoping a 2 ms window was enough.

In [8]:
# ================= COMPILE PROGRAM =================
def compile_program(rows):
    table = []
    tx_chunks = {lane: [] for lane in TX_LANES}
    tx_chunk_bounds = {lane: [0] for lane in TX_LANES}
    rx_total_s = {lane: 0.0 for lane in RX_LANES}
    refill_points = []   # (row_idx, lane, chunk_idx)
    lane_chunk_counter = {lane: 1 for lane in TX_LANES}

    for idx, row in enumerate(rows):
        mask = 0
        dur = {}
        for lane in TX_LANES:
            arr = row.get("tx", {}).get(lane)
            if arr is not None:
                mask |= (1 << LANE_BIT[lane])
                dur_s = len(arr) / SAMPLES_PER_BEAT / AXIS_BEAT_HZ
                dur[lane] = beats_for(dur_s)
                tx_chunks[lane].append(arr)
        for lane in RX_LANES:
            dur_s = row.get("rx", {}).get(lane)
            if dur_s:
                mask |= (1 << LANE_BIT[lane])
                dur[lane] = beats_for(dur_s)
                rx_total_s[lane] += dur_s
        for lane in row.get("refill", []):
            refill_points.append((idx, lane, lane_chunk_counter[lane]))
            lane_chunk_counter[lane] += 1
            tx_chunk_bounds[lane].append(sum(len(a) for a in tx_chunks[lane]))
        table.append({"mask": mask, "gap_cycles": seq_cycles_for(row["gap_s"]), "dur": dur})

    tx_full = {lane: (np.concatenate(chunks) if chunks else np.zeros(0, dtype=np.complex128))
               for lane, chunks in tx_chunks.items()}
    for lane in TX_LANES:
        tx_chunk_bounds[lane].append(len(tx_full[lane]))

        # ---------------------------------------------------------
    # Exact sequencer row FIRE times.
    #
    # GAP belongs BEFORE the corresponding row fires:
    #
    #   fire_time[0] = gap[0]
    #   fire_time[1] = gap[0] + gap[1]
    #   ...
    #
    # All times are relative to the accepted start of the
    # sequencer run (apart from tiny fixed CDC/trigger latency,
    # irrelevant to millisecond refill budgeting).
    # ---------------------------------------------------------

    row_fire_s = []

    t = 0.0

    for row in table:
        t += row["gap_cycles"] / SEQ_CLK_HZ
        row_fire_s.append(t)


    # ---------------------------------------------------------
    # Refill semantics:
    #
    # A refill marker on row R means:
    #
    #   "refill this lane during the GAP BEFORE row R fires."
    #
    # The new chunk's hard deadline is the firing time of the
    # next row that actually consumes that TX lane.
    #
    # start_try_s is the beginning of row R's pre-fire GAP:
    #
    #   row 0 -> t=0
    #   row R -> fire time of row R-1
    # ---------------------------------------------------------

    refill_deadlines = []

    for row_idx, lane, chunk_idx in refill_points:

        start_try_s = (
            0.0
            if row_idx == 0
            else row_fire_s[row_idx - 1]
        )

        deadline_s = None
        consume_row = None

        for j in range(row_idx, len(rows)):

            if lane in rows[j].get("tx", {}):
                deadline_s = row_fire_s[j]
                consume_row = j
                break

        refill_deadlines.append(
            (
                row_idx,
                lane,
                chunk_idx,
                start_try_s,
                deadline_s,
                consume_row,
            )
        )

    return (
        table,
        tx_full,
        tx_chunk_bounds,
        refill_deadlines,
        rx_total_s,
        row_fire_s,
    )

table, tx_full, tx_chunk_bounds, refill_deadlines, rx_total_s, row_fire_s = compile_program(rows)

if len(table) > 256:
    raise ValueError(f"Program has {len(table)} rows, exceeds the 256-row Pulse_Sequencer build.")

for lane in TX_LANES:
    print(f"{lane}: {len(tx_full[lane]):,} samples total across the whole program")
for lane, secs in rx_total_s.items():
    if secs > CAP_MAX_S:
        raise RuntimeError(f"{lane}: total capture {secs*1e3:.1f} ms exceeds the {CAP_MAX_S*1e3:.0f} ms RX DMA limit.")
    print(f"{lane}: {secs*1e3:.3f} ms total capture this program")

print()
print("Refill schedule:")

for (
    row_idx,
    lane,
    chunk_idx,
    start_try_s,
    deadline_s,
    consume_row,
) in refill_deadlines:

    window_s = (
        None
        if deadline_s is None
        else deadline_s - start_try_s
    )

    print(
        f"  gap before row {row_idx:>3}: "
        f"load {lane} chunk {chunk_idx}; "
        f"consume row={consume_row}, "
        f"try_from={start_try_s*1e3:.4f} ms, "
        f"deadline={deadline_s*1e3:.4f} ms, "
        f"available={window_s*1e3:.4f} ms"
    )


for lane in TX_LANES:
    bounds = tx_chunk_bounds[lane]
    for c0, c1 in zip(bounds[:-1], bounds[1:]):
        chunk_bytes = (c1 - c0) * 4
        if chunk_bytes > DMA_MAX_BYTES:
            raise RuntimeError(f"{lane}: a chunk between samples {c0}-{c1} is {chunk_bytes:,} bytes, "
                                f"exceeds the {DMA_MAX_BYTES:,}-byte single-DMA-transfer limit.")
    chunk_sizes = [(b1-b0)*4 for b0, b1 in zip(bounds[:-1], bounds[1:])]
    print(f"{lane}: {len(bounds)-1} chunk(s), sizes(bytes)={chunk_sizes}")

print("PASS: program compiled and passes capacity/consistency checks.")


TX_A: 4,921,840 samples total across the whole program
TX_B: 49,152,000 samples total across the whole program
RX_A: 0.054 ms total capture this program
RX_B: 0.054 ms total capture this program

Refill schedule:
  gap before row   2: load TX_B chunk 1; consume row=3, try_from=22.0100 ms, deadline=44.0200 ms, available=22.0100 ms
  gap before row   4: load TX_B chunk 2; consume row=5, try_from=44.0200 ms, deadline=66.0300 ms, available=22.0100 ms
  gap before row   6: load TX_B chunk 3; consume row=7, try_from=66.0300 ms, deadline=88.0400 ms, available=22.0100 ms
  gap before row   8: load TX_B chunk 4; consume row=9, try_from=88.0400 ms, deadline=110.0500 ms, available=22.0100 ms
  gap before row  10: load TX_B chunk 5; consume row=11, try_from=110.0500 ms, deadline=132.0600 ms, available=22.0100 ms
  gap before row  12: load TX_B chunk 6; consume row=13, try_from=132.0600 ms, deadline=154.0700 ms, available=22.0100 ms
  gap before row  14: load TX_B chunk 7; consume row=15, try_from=

## Cell 9 -- Standalone `transfer()` call-overhead benchmark (run before anything is armed)

Measures pure Python/PYNQ/AXI-Lite overhead of issuing a refill-sized `transfer()` call, with
the sequencer not even running (so nothing drains the buffer and this is purely the call
itself). This replaces the placeholder `TRANSFER_CALL_OVERHEAD_S_ESTIMATE` guess with a real
number for this board. `ch.stop()` resets the channel to halted between iterations so each
call starts from the same state `_try_refill()` would see mid-run.

In [9]:
# ================= ONE-ROW TX-B EXTERNAL-MODE QUALIFICATION =================
#
# Controlled A/B against the INTERNAL-mode test that just PASSED.
#
# SAME:
#   - 256-row bitstream
#   - TX-B
#   - 20 ms waveform
#   - memory bank 1
#   - DMA pre-armed before TTL_ARM
#   - Pulse Sequencer hardware trigger
#   - AW15 -> AW16 loopback
#
# DIFFERENCE:
#   DUR0 = TEST_BEATS instead of 0
#
# Therefore:
#   hw_active_length != 0
#   -> TX_Multi_Gate EXTERNAL mode
#
# No Ethernet.
# No RX.
# No refills.
# No SW_START.
# No dependence on TX internal table.


# ============================================================
# 0. DIAGNOSTIC CONSTANTS -- all test-specific values here
# ============================================================

TEST_TX_S = 20e-3

# Keep the first-row delay short so the timing is easy to see.
TEST_ROW_GAP_S = 1e-3

DMA_ARM_SETTLE_S = 1e-3
POLL_S = 100e-6
PRINT_INTERVAL_S = 2e-3
TEST_TIMEOUT_S = 1.0

TEST_BEATS = beats_for(TEST_TX_S)
TEST_ROW_GAP_CYCLES = seq_cycles_for(TEST_ROW_GAP_S)


print("=== ONE-ROW TX-B EXTERNAL-MODE TEST ===")
print(f"TX duration:  {TEST_TX_S*1e3:.3f} ms")
print(f"TX beats:     {TEST_BEATS}")
print(f"Row gap:      {TEST_ROW_GAP_S*1e3:.3f} ms")
print(f"Gap cycles:   {TEST_ROW_GAP_CYCLES}")
print()
print("Expected rough timing:")
print("  ~1 ms  : sequencer row fires")
print("  ~1 ms  : TX-B becomes BUSY")
print("  ~21 ms : TX-B closes and DMA becomes IDLE")


# ============================================================
# 1. BUILD EXACTLY ONE 20-ms TX-B WAVEFORM
# ============================================================

test_wave = chirp(
    5,
    25,
    TEST_TX_S,
    DAC_B_NCO_MHZ
)

test_data = quantize_iq(test_wave)

expected_samples = samples_for(TEST_BEATS)

if len(test_wave) != expected_samples:
    raise RuntimeError(
        "Waveform length mismatch: "
        f"{len(test_wave):,} complex samples vs "
        f"{expected_samples:,} expected"
    )


print()
print(f"Waveform complex samples: {len(test_wave):,}")
print(f"DMA int16 elements:       {len(test_data):,}")
print(f"DMA bytes:                {test_data.nbytes:,}")


# ============================================================
# 2. DMA LENGTH SANITY CHECK
# ============================================================

DMA_MAX_BYTES_TEST = (1 << 26) - 1

if test_data.nbytes > DMA_MAX_BYTES_TEST:
    raise RuntimeError(
        f"Test buffer is {test_data.nbytes:,} bytes, "
        f"which exceeds the configured 26-bit DMA limit "
        f"of {DMA_MAX_BYTES_TEST:,} bytes"
    )


# ============================================================
# 3. ALLOCATE TX BUFFER IN THE KNOWN-GOOD TX MEMORY BANK
#
# IMPORTANT:
# This intentionally mirrors the working Ethernet notebook.
# Do NOT replace this with ordinary allocate().
# ============================================================

test_buf = base.device.get_memory_by_idx(1).allocate(
    shape=test_data.shape,
    dtype=np.int16
)

test_buf[:] = test_data
test_buf.flush()

test_pa = int(test_buf.physical_address)


print()
print(
    "TX-B test buffer:",
    f"PA=0x{test_pa:x}",
    f"bytes={test_buf.nbytes:,}"
)


if test_pa < 0x1000000000:
    raise RuntimeError(
        "TX-B test buffer is not in the known-good TX memory region: "
        f"PA=0x{test_pa:x}"
    )


if test_buf.nbytes != test_data.nbytes:
    raise RuntimeError(
        f"TX-B buffer size mismatch: "
        f"{test_buf.nbytes:,} vs {test_data.nbytes:,}"
    )


# ============================================================
# 4. LOCAL DMA HELPERS
# ============================================================

def ext_dma_status(ch):
    return int(
        ch._mmio.read(
            int(ch._offset) + 0x04
        )
    )


def ext_dma_flags(st):

    names = []

    if st & DMASR_HALTED:
        names.append("HALTED")

    if st & DMASR_IDLE:
        names.append("IDLE")

    if st & DMASR_ERR_MASK:
        names.append("ERROR")

    if not names:
        names.append("RUNNING/NON-IDLE")

    return "|".join(names)


def ext_dump(label):

    seq_ptr = int(
        seq.mmio.read(SEQ_ROW_PTR)
    )

    ttl_st = int(
        seq.mmio.read(SEQ_TTL_STATUS)
    )

    edge_count = int(
        seq.mmio.read(SEQ_TTL_EDGE_COUNT)
    )

    cdc = int(
        seq.mmio.read(SEQ_CDC_OVERRUN)
    ) & 0xF

    tx_st = int(
        tx_b.mmio.read(TXM_STATUS)
    )

    tx_seg = int(
        tx_b.mmio.read(TXM_SEG_PTR)
    )

    dma_st = ext_dma_status(
        dma_tb.sendchannel
    )

    print()
    print(f"--- {label} ---")

    print(
        f"SEQ: "
        f"ROW_PTR={seq_ptr} "
        f"TTL_STATUS=0x{ttl_st:08x} "
        f"EDGE_COUNT={edge_count} "
        f"CDC_OVERRUN=0x{cdc:x}"
    )

    print(
        f"TX_B: "
        f"STATUS=0x{tx_st:08x} "
        f"BUSY={bool(tx_st & TX_BUSY)} "
        f"CONTROL_OVERRUN={bool(tx_st & TX_OVERRUN)} "
        f"SEG_PTR={tx_seg}"
    )

    print(
        f"DMA_TX_B: "
        f"0x{dma_st:08x} "
        f"({ext_dma_flags(dma_st)})"
    )


# ============================================================
# 5. CLEAN SEQUENCER STATE
#
# Same reset/programming order as the known-working path.
# ============================================================

seq.mmio.write(
    SEQ_TTL_MODE,
    0
)

seq.mmio.write(
    SEQ_ENABLE,
    0
)

seq.mmio.write(
    SEQ_RESET,
    1
)

time.sleep(100e-6)

seq.mmio.write(
    SEQ_CDC_OVERRUN,
    0xF
)

seq.mmio.write(
    SEQ_TTL_STATUS_CLEAR,
    1
)


# Clear TX-B sticky CONTROL_OVERRUN.
tx_b.mmio.write(
    TXM_STATUS,
    TX_OVERRUN
)

time.sleep(100e-6)


# ============================================================
# 6. PROGRAM EXACTLY ONE PULSE-SEQUENCER ROW
#
# TX-B = lane 0
#
# CRITICAL A/B VARIABLE:
#
#     DUR0 = TEST_BEATS
#
# rather than:
#
#     DUR0 = 0
#
# Nonzero hw_active_length selects EXTERNAL mode.
# ============================================================

seq.mmio.write(
    SEQ_ROW_SEL,
    0
)

seq.mmio.write(
    SEQ_MASK,
    1 << LANE_TX_B
)

seq.mmio.write(
    SEQ_GAP,
    TEST_ROW_GAP_CYCLES
)

# -------------------------------
# EXTERNAL MODE
# -------------------------------
seq.mmio.write(
    SEQ_DUR0,
    int(TEST_BEATS)
)

# Other lanes disabled.
seq.mmio.write(
    SEQ_DUR1,
    0
)

seq.mmio.write(
    SEQ_DUR2,
    0
)

seq.mmio.write(
    SEQ_DUR3,
    0
)

seq.mmio.write(
    SEQ_COMMIT,
    1
)

seq.mmio.write(
    SEQ_TABLE_LEN,
    1
)

# Clear any stale CDC flag after programming.
seq.mmio.write(
    SEQ_CDC_OVERRUN,
    0xF
)

time.sleep(100e-6)


# ============================================================
# 7. ENABLE TTL MODE
# ============================================================

seq.mmio.write(
    SEQ_TTL_MODE,
    1
)

time.sleep(100e-6)


ext_dump(
    "after EXTERNAL sequencer programming"
)


# ============================================================
# 8. GET TX-B DMA CHANNEL
# ============================================================

ch = dma_tb.sendchannel

dma_st = ext_dma_status(ch)


if dma_st & DMASR_ERR_MASK:

    raise RuntimeError(
        "TX-B DMA already contains an error before transfer: "
        f"0x{dma_st:08x} ({ext_dma_flags(dma_st)})"
    )


# ============================================================
# 9. START DMA IF HALTED
#
# Bounded direct RS write; no unbounded ch.start().
# ============================================================

if dma_st & DMASR_HALTED:

    print()
    print("TX-B DMA is HALTED; starting it...")

    ch._mmio.write(
        ch._offset,
        0x0001
    )

    start_t0 = time.perf_counter()

    while True:

        dma_st = ext_dma_status(ch)

        if dma_st & DMASR_ERR_MASK:
            raise RuntimeError(
                "TX-B DMA error while starting: "
                f"0x{dma_st:08x}"
            )

        if not (dma_st & DMASR_HALTED):
            break

        if (
            time.perf_counter() - start_t0
            > 0.5
        ):
            raise TimeoutError(
                "TX-B DMA did not enter RUN state: "
                f"0x{dma_st:08x}"
            )

        time.sleep(POLL_S)


print()
print(
    "TX-B DMA before transfer:",
    f"0x{ext_dma_status(ch):08x}",
    ext_dma_flags(
        ext_dma_status(ch)
    )
)


# ============================================================
# 10. PRE-ARM DMA BEFORE TTL_ARM
#
# Same ordering as the known-working Ethernet notebook.
# ============================================================

print()
print("Arming TX-B DMA...")


ch.transfer(
    test_buf
)


time.sleep(
    DMA_ARM_SETTLE_S
)


dma_st = ext_dma_status(ch)


print(
    "TX-B DMA after transfer():",
    f"0x{dma_st:08x}",
    ext_dma_flags(dma_st)
)


if dma_st & DMASR_ERR_MASK:

    raise RuntimeError(
        "TX-B DMA error immediately after transfer(): "
        f"0x{dma_st:08x}"
    )


if dma_st & DMASR_HALTED:

    raise RuntimeError(
        "TX-B DMA unexpectedly HALTED after transfer(): "
        f"0x{dma_st:08x}"
    )


ext_dump(
    "DMA armed, BEFORE TTL_ARM"
)


# ============================================================
# 11. TTL ARM
#
# AW15 -> AW16 loopback should create the accepted edge.
# ============================================================

print()
print("Writing TTL_ARM...")


seq.mmio.write(
    SEQ_TTL_ARM,
    1
)


# ============================================================
# 12. MONITOR THE ENTIRE TX OPERATION
#
# IMPORTANT:
#
# We DO NOT stop when RUN_DONE appears.
#
# The sequencer's RUN_DONE means its one-row schedule has
# completed. It does NOT mean TX-B has finished transmitting.
#
# Success requires:
#
#   TTL_RUN_DONE
#   DMA IDLE
#   TX_B not BUSY
# ============================================================

t0 = time.perf_counter()

saw_edge = False
saw_running = False
saw_tx_busy = False
saw_dma_idle = False

tx_busy_first_s = None
tx_busy_last_s = None
dma_idle_first_s = None
run_done_first_s = None

last_print = -1.0


while True:

    now = time.perf_counter()
    elapsed = now - t0


    ttl_st = int(
        seq.mmio.read(
            SEQ_TTL_STATUS
        )
    )

    edge_count = int(
        seq.mmio.read(
            SEQ_TTL_EDGE_COUNT
        )
    )

    row_ptr = int(
        seq.mmio.read(
            SEQ_ROW_PTR
        )
    )

    cdc = int(
        seq.mmio.read(
            SEQ_CDC_OVERRUN
        )
    ) & 0xF

    tx_st = int(
        tx_b.mmio.read(
            TXM_STATUS
        )
    )

    tx_seg = int(
        tx_b.mmio.read(
            TXM_SEG_PTR
        )
    )

    dma_st = ext_dma_status(ch)


    # --------------------------------------------------------
    # Observations
    # --------------------------------------------------------

    if edge_count:
        saw_edge = True


    if ttl_st & TTL_RUNNING:
        saw_running = True


    if tx_st & TX_BUSY:

        saw_tx_busy = True

        if tx_busy_first_s is None:
            tx_busy_first_s = elapsed

        tx_busy_last_s = elapsed


    if dma_st & DMASR_IDLE:

        saw_dma_idle = True

        if dma_idle_first_s is None:
            dma_idle_first_s = elapsed


    if ttl_st & TTL_RUN_DONE:

        if run_done_first_s is None:
            run_done_first_s = elapsed


    # --------------------------------------------------------
    # Periodic status print
    # --------------------------------------------------------

    if (
        last_print < 0
        or
        elapsed - last_print >= PRINT_INTERVAL_S
    ):

        print(
            f"t={elapsed*1e3:8.3f} ms | "
            f"TTL=0x{ttl_st:x} "
            f"EDGE={edge_count} "
            f"ROW={row_ptr} "
            f"CDC=0x{cdc:x} "
            f"TXB=0x{tx_st:x} "
            f"SEG={tx_seg} "
            f"DMA=0x{dma_st:08x}"
        )

        last_print = elapsed


    # --------------------------------------------------------
    # Hard failures
    # --------------------------------------------------------

    if dma_st & DMASR_ERR_MASK:

        raise RuntimeError(
            "TX-B DMA error during EXTERNAL test: "
            f"0x{dma_st:08x} "
            f"({ext_dma_flags(dma_st)})"
        )


    if cdc != 0:

        raise RuntimeError(
            "Pulse Sequencer CDC_OVERRUN during "
            f"EXTERNAL test: 0x{cdc:x}"
        )


    if tx_st & TX_OVERRUN:

        raise RuntimeError(
            "TX-B CONTROL_OVERRUN during EXTERNAL test: "
            f"STATUS=0x{tx_st:08x}"
        )


    # --------------------------------------------------------
    # SUCCESS CONDITION
    #
    # Sequencer done
    # +
    # DMA drained
    # +
    # TX gate closed
    # --------------------------------------------------------

    if (
        (ttl_st & TTL_RUN_DONE)
        and
        (dma_st & DMASR_IDLE)
        and
        not (tx_st & TX_BUSY)
    ):
        break


    # --------------------------------------------------------
    # Overall timeout
    # --------------------------------------------------------

    if elapsed > TEST_TIMEOUT_S:

        print()
        print(
            "TIMEOUT waiting for "
            "RUN_DONE + DMA IDLE + TX-B CLOSED."
        )

        break


    time.sleep(
        POLL_S
    )


# ============================================================
# 13. FINAL STATE
# ============================================================

ext_dump(
    "FINAL EXTERNAL-MODE TEST"
)


final_ttl = int(
    seq.mmio.read(
        SEQ_TTL_STATUS
    )
)

final_edge = int(
    seq.mmio.read(
        SEQ_TTL_EDGE_COUNT
    )
)

final_cdc = int(
    seq.mmio.read(
        SEQ_CDC_OVERRUN
    )
) & 0xF

final_tx = int(
    tx_b.mmio.read(
        TXM_STATUS
    )
)

final_dma = ext_dma_status(ch)


# ============================================================
# 14. TIMING SUMMARY
# ============================================================

print()
print("Timing observations:")


if tx_busy_first_s is None:
    print("  TX-B first BUSY:     never observed")
else:
    print(
        f"  TX-B first BUSY:     "
        f"{tx_busy_first_s*1e3:.3f} ms"
    )


if tx_busy_last_s is None:
    print("  TX-B last BUSY poll: never observed")
else:
    print(
        f"  TX-B last BUSY poll: "
        f"{tx_busy_last_s*1e3:.3f} ms"
    )


if dma_idle_first_s is None:
    print("  DMA first IDLE:      never observed")
else:
    print(
        f"  DMA first IDLE:      "
        f"{dma_idle_first_s*1e3:.3f} ms"
    )


if run_done_first_s is None:
    print("  RUN_DONE:            never observed")
else:
    print(
        f"  RUN_DONE:            "
        f"{run_done_first_s*1e3:.3f} ms"
    )


# ============================================================
# 15. RESULT
# ============================================================

print()
print("Observed:")
print(f"  TTL edge:       {saw_edge}")
print(f"  sequencer RUN:  {saw_running}")
print(f"  TX-B BUSY:      {saw_tx_busy}")
print(f"  TX-B DMA IDLE:  {saw_dma_idle}")


print()
print(
    "========== EXTERNAL MODE RESULT =========="
)


if final_edge == 0:

    print(
        "FAIL: TTL edge was not accepted."
    )


elif final_cdc != 0:

    print(
        f"FAIL: CDC_OVERRUN=0x{final_cdc:x}"
    )


elif not saw_tx_busy:

    print(
        "FAIL: TX-B never entered ACTIVE/BUSY."
    )


elif not (
    final_dma & DMASR_IDLE
):

    print(
        "FAIL: TX-B entered EXTERNAL mode, "
        "but the DMA did not drain."
    )


elif final_tx & TX_BUSY:

    print(
        "FAIL: DMA drained but TX-B "
        "remained BUSY."
    )


elif not (
    final_ttl & TTL_RUN_DONE
):

    print(
        "FAIL: TX completed, but the "
        "sequencer did not assert RUN_DONE."
    )


else:

    print(
        "PASS: EXTERNAL mode works on "
        "the 256-row bitstream."
    )


print(
    "=========================================="
)


=== ONE-ROW TX-B EXTERNAL-MODE TEST ===
TX duration:  20.000 ms
TX beats:     307200
Row gap:      1.000 ms
Gap cycles:   100000

Expected rough timing:
  ~1 ms  : sequencer row fires
  ~1 ms  : TX-B becomes BUSY
  ~21 ms : TX-B closes and DMA becomes IDLE

Waveform complex samples: 2,457,600
DMA int16 elements:       4,915,200
DMA bytes:                9,830,400

TX-B test buffer: PA=0x1000000000 bytes=9,830,400

--- after EXTERNAL sequencer programming ---
SEQ: ROW_PTR=0 TTL_STATUS=0x00000000 EDGE_COUNT=0 CDC_OVERRUN=0x0
TX_B: STATUS=0x00000000 BUSY=False CONTROL_OVERRUN=False SEG_PTR=0
DMA_TX_B: 0x00000000 (RUNNING/NON-IDLE)

TX-B DMA before transfer: 0x00000000 RUNNING/NON-IDLE

Arming TX-B DMA...
TX-B DMA after transfer(): 0x00000000 RUNNING/NON-IDLE

--- DMA armed, BEFORE TTL_ARM ---
SEQ: ROW_PTR=0 TTL_STATUS=0x00000000 EDGE_COUNT=0 CDC_OVERRUN=0x0
TX_B: STATUS=0x00000000 BUSY=False CONTROL_OVERRUN=False SEG_PTR=0
DMA_TX_B: 0x00000000 (RUNNING/NON-IDLE)

Writing TTL_ARM...
t=   3

## Cell 10 -- RFDC NCO setup + DMA buffers (one buffer per TX lane, sliced for chunks)

The fix for "why 20 buffers": each lane's whole waveform is quantized once into a single PL
buffer; refill chunks are slices of that one buffer, not separate allocations. RX is a single
buffer per lane (no double buffering -- nothing to overlap without Ethernet).

In [10]:
# ================= RFDC NCO SETUP =================
for dac, freq in ((dac_b, DAC_B_NCO_MHZ), (dac_a, DAC_A_NCO_MHZ)):
    ms = dac.MixerSettings
    ms['Freq'] = float(freq)
    ms['PhaseOffset'] = 0.0
    ms['EventSource'] = 2
    dac.MixerSettings = ms
    dac.UpdateEvent(xrfdc.EVENT_MIXER)
for dac, freq in ((dac_b, DAC_B_NCO_MHZ), (dac_a, DAC_A_NCO_MHZ)):
    assert abs(float(dac.MixerSettings['Freq']) - freq) < 1e-6
print('PASS: DAC mixer state matches config.')

# ================= DMA BUFFERS (single allocation per lane) =================
tx_full_buf = {}     # lane -> one PynqBuffer holding the WHOLE quantized waveform
tx_chunk_view = {}    # lane -> list of slices (views) into tx_full_buf[lane], one per chunk
for lane in TX_LANES:
    quantized = quantize_iq(tx_full[lane])           # one quantize_iq() call for the whole lane
    buf = base.device.get_memory_by_idx(1).allocate(shape=quantized.shape, dtype=np.int16)
    buf[:] = quantized
    buf.flush()                                       # one flush for the whole buffer
    tx_full_buf[lane] = buf

    bounds = tx_chunk_bounds[lane]
    views = [buf[2*c0:2*c1] for c0, c1 in zip(bounds[:-1], bounds[1:])]
    tx_chunk_view[lane] = views

    # Sanity-check the key assumption this whole design rests on: that a slice of a PynqBuffer
    # reports the correct physical address (base + byte offset), so transfer()ing a slice
    # actually points the DMA at the right bytes instead of silently re-sending chunk 0.
    base_pa = int(buf.physical_address)
    for k, (c0, c1) in enumerate(zip(bounds[:-1], bounds[1:])):
        expected_pa = base_pa + 2*c0*2   # 2 int16/sample * 2 bytes/int16
        actual_pa = int(views[k].physical_address)
        if actual_pa != expected_pa:
            raise RuntimeError(f"{lane} chunk {k}: slice physical_address 0x{actual_pa:x} != "
                                f"expected 0x{expected_pa:x}. Buffer slicing does not behave as "
                                f"assumed on this PYNQ version -- do not trust sliced transfers; "
                                f"fall back to one allocate() per chunk instead.")
    print(f"{lane}: 1 buffer allocated ({buf.nbytes:,} bytes), {len(views)} chunk view(s), "
          f"physical addresses verified.")

rx_buffers = {}
for lane in RX_LANES:
    capture_beats = beats_for(rx_total_s[lane])
    capture_samples = samples_for(capture_beats)
    rx_buffers[lane] = allocate(shape=(capture_samples,), dtype=np.int16)
    print(f"{lane}: RX buffer {rx_buffers[lane].nbytes:,} bytes (single, reused every shot).")

print('PASS: DMA buffers allocated.')


PASS: DAC mixer state matches config.
TX_A: 1 buffer allocated (19,687,360 bytes), 1 chunk view(s), physical addresses verified.
TX_B: 1 buffer allocated (196,608,000 bytes), 20 chunk view(s), physical addresses verified.
RX_A: RX buffer 13,280 bytes (single, reused every shot).
RX_B: RX buffer 13,280 bytes (single, reused every shot).
PASS: DMA buffers allocated.


## Cell 11 -- Commit the compiled table to the sequencer

In [11]:
# ================= COMMIT TABLE TO HARDWARE =================
def commit_table(table):
    seq.mmio.write(SEQ_TTL_MODE, 0)
    seq.mmio.write(SEQ_ENABLE, 0)
    seq.mmio.write(SEQ_RESET, 1)
    time.sleep(100e-6)
    for idx, row in enumerate(table):
        seq.mmio.write(SEQ_ROW_SEL, idx)
        seq.mmio.write(SEQ_MASK, row["mask"])
        seq.mmio.write(SEQ_GAP, row["gap_cycles"])
        seq.mmio.write(SEQ_DUR0, row["dur"].get("TX_B", 0))
        seq.mmio.write(SEQ_DUR1, row["dur"].get("TX_A", 0))
        seq.mmio.write(SEQ_DUR2, row["dur"].get("RX_B", 0))
        seq.mmio.write(SEQ_DUR3, row["dur"].get("RX_A", 0))
        seq.mmio.write(SEQ_COMMIT, 1)
    seq.mmio.write(SEQ_TABLE_LEN, len(table))
    print(f"PASS: committed {len(table)} rows to the sequencer.")

commit_table(table)


PASS: committed 42 rows to the sequencer.


## Cell 12 -- DMA/loop helpers + `dump_state()` (no Ethernet, no threading)

In [12]:
# ================= DMA / LOOP HELPERS =================
def dma_status(ch): return int(ch._mmio.read(int(ch._offset)+0x04))
def dma_flags(v):
    out = ['halted' if v&1 else 'running']
    if v&2: out.append('idle')
    if v&0x10: out.append('INTERNAL_ERR')
    if v&0x20: out.append('SLAVE_ERR')
    if v&0x40: out.append('DECODE_ERR')
    return '|'.join(out)

def ensure_running(ch, label):
    st = dma_status(ch)

    if st & DMASR_ERR_MASK:
        raise RuntimeError(
            f"{label}: DMA error before shot: "
            f"0x{st:08x} ({dma_flags(st)})"
        )

    if st & DMASR_HALTED:
        safe_start(ch, label, timeout_s=0.2)

        st = dma_status(ch)

        if st & DMASR_ERR_MASK:
            raise RuntimeError(
                f"{label}: DMA error after restart: "
                f"0x{st:08x} ({dma_flags(st)})"
            )

        if st & DMASR_HALTED:
            raise RuntimeError(
                f"{label}: still HALTED after restart: "
                f"0x{st:08x} ({dma_flags(st)})"
            )

def wait_dma_idle(ch, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = dma_status(ch)
        if st & DMASR_ERR_MASK: raise RuntimeError(f'{label}: DMA error: 0x{st:08x} ({dma_flags(st)})')
        if st & DMASR_IDLE: return
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: DMA not idle: 0x{st:08x} ({dma_flags(st)})')
        time.sleep(50e-6)

def wait_cap_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(CAP_STATUS))
        if not (st & CAP_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: Capture_Gate busy: 0x{st:08x}')
        time.sleep(100e-6)

def wait_tx_idle(g, label, timeout_s):
    t0 = time.perf_counter()
    while True:
        st = int(g.mmio.read(TXM_STATUS))
        if not (st & TX_BUSY): return st
        if time.perf_counter()-t0 > timeout_s: raise TimeoutError(f'{label}: TX gate busy: 0x{st:08x}')
        time.sleep(20e-6)

def discard_rx(buf):
    inv = getattr(buf, 'invalidate', None)
    if inv:
        try: inv()
        except Exception: pass

def ttl_arm(s): s.mmio.write(SEQ_TTL_ARM, 1)

def dump_state(label=""):
    print(f"---- dump_state: {label} ----")
    try:
        print(f"  seq:   ROW_PTR={int(seq.mmio.read(SEQ_ROW_PTR))} "
              f"TTL_STATUS=0x{int(seq.mmio.read(SEQ_TTL_STATUS)):x} "
              f"CDC_OVERRUN=0x{int(seq.mmio.read(SEQ_CDC_OVERRUN))&0xF:x}")
    except Exception as e:
        print(f"  seq: <read failed: {e}>")
    for name, g in [("tx_a", tx_a), ("tx_b", tx_b)]:
        try:
            st = int(g.mmio.read(TXM_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&TX_BUSY)}, overrun={bool(st&TX_OVERRUN)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, g in [("cap_a", cap_a), ("cap_b", cap_b)]:
        try:
            st = int(g.mmio.read(CAP_STATUS))
            print(f"  {name}: STATUS=0x{st:x} (busy={bool(st&CAP_BUSY)}, overflow={bool(st&CAP_OVERFLOW)})")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    for name, ch in [("RX-A", dma_ra.recvchannel), ("RX-B", dma_rb.recvchannel),
                      ("TX-A", dma_ta.sendchannel), ("TX-B", dma_tb.sendchannel)]:
        try:
            st = dma_status(ch)
            print(f"  {name}: 0x{st:08x} ({dma_flags(st)}) transferred={int(ch.transferred)}")
        except Exception as e:
            print(f"  {name}: <read failed: {e}>")
    print("-" * (18 + len(label)))

print('DMA/loop helpers ready.')


DMA/loop helpers ready.


## Cell 13 -- Run loop

Synchronous, single-buffered, no Ethernet. Every refill logs the *measured* margin: modeled
deadline (from Cell 8) minus actual wall-clock time the `transfer()` call returned, both
relative to this shot's own `TTL_ARM`. A negative margin means that refill would have missed
its window -- exactly the failure mode from the earlier run, now visible instead of inferred.

In [13]:
# ================= RUN LOOP =================

def prepare_shot():
    wait_cap_idle(cap_a, "RX_A", 2.0)
    wait_cap_idle(cap_b, "RX_B", 2.0)

    wait_tx_idle(tx_a, "TX_A", 1.0)
    wait_tx_idle(tx_b, "TX_B", 1.0)

    cap_a.mmio.write(CAP_CLEAR, CAP_OVERFLOW)
    cap_b.mmio.write(CAP_CLEAR, CAP_OVERFLOW)

    cap_a.mmio.write(
        CAP_LENGTH,
        beats_for(rx_total_s["RX_A"])
    )

    cap_b.mmio.write(
        CAP_LENGTH,
        beats_for(rx_total_s["RX_B"])
    )

    tx_a.mmio.write(TXM_STATUS, TX_OVERRUN)
    tx_b.mmio.write(TXM_STATUS, TX_OVERRUN)

    seq.mmio.write(SEQ_ENABLE, 0)
    seq.mmio.write(SEQ_RESET, 1)

    time.sleep(100e-6)

    seq.mmio.write(SEQ_CDC_OVERRUN, 0xF)
    seq.mmio.write(SEQ_TTL_STATUS_CLEAR, 1)
    seq.mmio.write(SEQ_TTL_MODE, 1)

    for ch, label in [
        (dma_ra.recvchannel, "RX-A"),
        (dma_rb.recvchannel, "RX-B"),
        (dma_ta.sendchannel, "TX-A"),
        (dma_tb.sendchannel, "TX-B"),
    ]:
        ensure_running(ch, label)


def arm_shot():

    # RX DMAs
    dma_ra.recvchannel.transfer(
        rx_buffers["RX_A"]
    )

    dma_rb.recvchannel.transfer(
        rx_buffers["RX_B"]
    )

    # Initial TX chunks.
    #
    # Later chunks are loaded by the refill scheduler below.
    dma_ta.sendchannel.transfer(
        tx_chunk_view["TX_A"][0]
    )

    dma_tb.sendchannel.transfer(
        tx_chunk_view["TX_B"][0]
    )

    # Conservative initial DMA-arm settle.
    time.sleep(5e-3)

    for ch, label in [
        (dma_ra.recvchannel, "RX-A"),
        (dma_rb.recvchannel, "RX-B"),
        (dma_ta.sendchannel, "TX-A"),
        (dma_tb.sendchannel, "TX-B"),
    ]:

        st = dma_status(ch)

        if st & (
            DMASR_ERR_MASK |
            DMASR_HALTED
        ):
            raise RuntimeError(
                f"{label}: bad after arm "
                f"0x{st:08x} "
                f"({dma_flags(st)})"
            )


DMA_LANE_TO_CHANNEL = {
    "TX_A": dma_ta.sendchannel,
    "TX_B": dma_tb.sendchannel,
}


def _try_refill(
    row_idx,
    lane,
    chunk_idx,
    start_try_s,
    deadline_s,
    consume_row,
    t_run,
    margins,
):

    if chunk_idx >= len(
        tx_chunk_view[lane]
    ):
        return True

    ch = DMA_LANE_TO_CHANNEL[lane]

    st = dma_status(ch)

    if st & DMASR_ERR_MASK:
        raise RuntimeError(
            f"{lane}: DMA error before refill "
            f"(gap before row {row_idx}): "
            f"0x{st:08x} "
            f"({dma_flags(st)})"
        )

    # Previous transfer has not finished yet.
    #
    # This is NOT immediately an error. Keep polling until
    # either it becomes IDLE or the actual deadline arrives.
    if not (st & DMASR_IDLE):
        return False

    # DMA is now free: issue the next chunk.
    t_call_start = time.perf_counter()

    ch.transfer(
        tx_chunk_view[lane][chunk_idx]
    )

    t_call_done = time.perf_counter()

    elapsed_done_s = (
        t_call_done - t_run
    )

    call_s = (
        t_call_done - t_call_start
    )

    margin_s = (
        deadline_s - elapsed_done_s
        if deadline_s is not None
        else None
    )

    if margin_s is not None:
        margins.append(margin_s)

    if DEBUG_VERBOSE:

        if margin_s is None:
            margin_text = "deadline=n/a"
        else:
            flag = (
                ""
                if margin_s >= 0
                else " <<< MISSED DEADLINE"
            )

            margin_text = (
                f"margin={margin_s*1e3:+.4f} ms"
                f"{flag}"
            )

        print(
            f"  refill {lane} chunk {chunk_idx}: "
            f"gap-before-row={row_idx}, "
            f"consume-row={consume_row}, "
            f"issued={elapsed_done_s*1e3:.4f} ms, "
            f"transfer_call={call_s*1e3:.4f} ms, "
            f"{margin_text}"
        )

    return True


def run_shot(k):

    t0 = time.perf_counter()

    try:
        prepare_shot()
        arm_shot()

    except Exception:
        dump_state(
            f"shot {k} prepare/arm failure"
        )
        raise


    # ---------------------------------------------------------
    # Copy refill schedule for this shot.
    #
    # Cell 8 now supplies:
    #
    # (
    #   refill_row,
    #   lane,
    #   chunk_idx,
    #   start_try_s,
    #   deadline_s,
    #   consume_row
    # )
    # ---------------------------------------------------------

    pending = [
        (
            ri,
            lane,
            ci,
            start_s,
            deadline_s,
            consume_row,
        )
        for (
            ri,
            lane,
            ci,
            start_s,
            deadline_s,
            consume_row,
        )
        in refill_deadlines
    ]


    # ---------------------------------------------------------
    # TTL_ARM defines our software t=0 reference.
    # ---------------------------------------------------------

    ttl_arm(seq)

    t_run = time.perf_counter()

    margins = []


    # Whole-shot watchdog.
    deadline_wall = (
        time.perf_counter() + 5.0
    )


    while True:

        now = time.perf_counter()

        elapsed_s = (
            now - t_run
        )


        # -----------------------------------------------------
        # Service every refill whose pre-row window has opened.
        #
        # IMPORTANT:
        #
        # We no longer wait for:
        #
        #     ROW_PTR >= refill_row
        #
        # because GAP occurs BEFORE that row fires.
        #
        # Cell 8 tells us exactly when that refill window opens.
        # -----------------------------------------------------

        while pending:

            (
                row_idx,
                lane,
                chunk_idx,
                start_try_s,
                deadline_s,
                consume_row,
            ) = pending[0]


            # Window hasn't opened yet.
            if elapsed_s < start_try_s:
                break


            # If we've already passed the hard deadline before
            # even issuing transfer(), that's a real miss.
            if (
                deadline_s is not None
                and
                elapsed_s >= deadline_s
            ):

                dump_state(
                    f"shot {k} refill deadline miss"
                )

                raise RuntimeError(
                    f"shot {k}: {lane} chunk "
                    f"{chunk_idx} missed refill "
                    f"deadline before consume row "
                    f"{consume_row}; "
                    f"now={elapsed_s*1e3:.4f} ms, "
                    f"deadline="
                    f"{deadline_s*1e3:.4f} ms"
                )


            issued = _try_refill(
                row_idx,
                lane,
                chunk_idx,
                start_try_s,
                deadline_s,
                consume_row,
                t_run,
                margins,
            )


            if issued:

                pending.pop(0)

                # Refresh elapsed time before looking at the
                # next pending operation.
                elapsed_s = (
                    time.perf_counter()
                    - t_run
                )

            else:

                # DMA still busy.
                #
                # Don't block the whole control loop here.
                # Come back on the next polling iteration.
                break


        # -----------------------------------------------------
        # Hardware safety/status checks
        # -----------------------------------------------------

        cdc = int(
            seq.mmio.read(
                SEQ_CDC_OVERRUN
            )
        ) & 0xF

        if cdc:

            dump_state(
                f"shot {k} CDC overrun"
            )

            raise RuntimeError(
                f"shot {k}: "
                f"CDC overrun 0x{cdc:x}"
            )


        txa_st = int(
            tx_a.mmio.read(TXM_STATUS)
        )

        txb_st = int(
            tx_b.mmio.read(TXM_STATUS)
        )

        if txa_st & TX_OVERRUN:

            dump_state(
                f"shot {k} TX-A control overrun"
            )

            raise RuntimeError(
                f"shot {k}: TX-A "
                f"CONTROL_OVERRUN "
                f"0x{txa_st:08x}"
            )


        if txb_st & TX_OVERRUN:

            dump_state(
                f"shot {k} TX-B control overrun"
            )

            raise RuntimeError(
                f"shot {k}: TX-B "
                f"CONTROL_OVERRUN "
                f"0x{txb_st:08x}"
            )


        ttl_status = int(
            seq.mmio.read(
                SEQ_TTL_STATUS
            )
        )


        if ttl_status & TTL_RUN_DONE:
            break


        if (
            time.perf_counter()
            > deadline_wall
        ):

            row_ptr = int(
                seq.mmio.read(
                    SEQ_ROW_PTR
                )
            )

            dump_state(
                f"shot {k} RUN_DONE timeout"
            )

            raise TimeoutError(
                f"shot {k}: RUN_DONE timeout, "
                f"row_ptr={row_ptr}"
            )


        # 100 us instead of the previous 500 us.
        #
        # These refill windows are millisecond-scale; there is
        # no reason to throw away half a millisecond of margin
        # on every scheduler iteration.
        time.sleep(100e-6)


    # ---------------------------------------------------------
    # Sequencer finished.
    #
    # There should NOT be any refill still pending whose chunk
    # was actually required during this run.
    # ---------------------------------------------------------

    if pending:

        dump_state(
            f"shot {k} ended with pending refills"
        )

        raise RuntimeError(
            f"shot {k}: RUN_DONE with "
            f"{len(pending)} refill(s) still pending: "
            f"{pending[:3]}"
        )


    seq.mmio.write(
        SEQ_TTL_STATUS_CLEAR,
        1
    )

    row = int(
        seq.mmio.read(
            SEQ_ROW_PTR
        )
    )


    # ---------------------------------------------------------
    # Wait for downstream hardware completion.
    #
    # RUN_DONE is sequencer completion, not necessarily DMA/
    # gate completion.
    # ---------------------------------------------------------

    tout = max(
        2.0,
        max(rx_total_s.values()) + 1.0
    )


    try:

        wait_cap_idle(
            cap_a,
            "RX_A",
            tout
        )

        wait_cap_idle(
            cap_b,
            "RX_B",
            tout
        )

        wait_dma_idle(
            dma_ra.recvchannel,
            "RX-A",
            tout
        )

        wait_dma_idle(
            dma_rb.recvchannel,
            "RX-B",
            tout
        )

        wait_dma_idle(
            dma_ta.sendchannel,
            "TX-A",
            2.0
        )

        wait_dma_idle(
            dma_tb.sendchannel,
            "TX-B",
            2.0
        )

        wait_tx_idle(
            tx_a,
            "TX-A",
            2.0
        )

        wait_tx_idle(
            tx_b,
            "TX-B",
            2.0
        )

    except Exception:

        dump_state(
            f"shot {k} post-run wait failure"
        )

        raise


    # PYNQ bookkeeping completion.
    dma_ra.recvchannel.wait()
    dma_rb.recvchannel.wait()
    dma_ta.sendchannel.wait()
    dma_tb.sendchannel.wait()


    # ---------------------------------------------------------
    # Capture status
    # ---------------------------------------------------------

    ca = int(
        cap_a.mmio.read(
            CAP_STATUS
        )
    )

    cb = int(
        cap_b.mmio.read(
            CAP_STATUS
        )
    )

    overflow = (
        bool(ca & CAP_OVERFLOW)
        or
        bool(cb & CAP_OVERFLOW)
    )


    discard_rx(
        rx_buffers["RX_A"]
    )

    discard_rx(
        rx_buffers["RX_B"]
    )


    # ---------------------------------------------------------
    # Final health checks
    # ---------------------------------------------------------

    final_cdc = int(
        seq.mmio.read(
            SEQ_CDC_OVERRUN
        )
    ) & 0xF

    if final_cdc:

        raise RuntimeError(
            f"shot {k}: post-run "
            f"CDC overrun 0x{final_cdc:x}"
        )


    final_txa = int(
        tx_a.mmio.read(
            TXM_STATUS
        )
    )

    final_txb = int(
        tx_b.mmio.read(
            TXM_STATUS
        )
    )

    if final_txa & TX_OVERRUN:
        raise RuntimeError(
            f"shot {k}: post-run "
            "TX-A CONTROL_OVERRUN"
        )

    if final_txb & TX_OVERRUN:
        raise RuntimeError(
            f"shot {k}: post-run "
            "TX-B CONTROL_OVERRUN"
        )


    return {
        "shot": k,
        "row": row,
        "overflow": overflow,
        "elapsed_s": (
            time.perf_counter() - t0
        ),
        "min_margin_s": (
            min(margins)
            if margins
            else None
        ),
    }


print(
    "Run-loop functions ready."
)


Run-loop functions ready.


## Cell 14 -- Run + summary

In [15]:
# ================= RUN =================
results = []
for k in range(1, LOOP_COUNT+1):
    try:
        r = run_shot(k); results.append(r)
        mm = r['min_margin_s']
        mm_str = f"{mm*1e3:+.4f} ms" if mm is not None else 'n/a'
        print(f"shot {k:04d}: row={r['row']} overflow={r['overflow']} "
              f"elapsed={r['elapsed_s']:.3f}s min_refill_margin={mm_str}")
    except Exception:
        print(f"shot {k} FAILED -- dumping hardware state:")
        dump_state(f"shot {k} exception")
        seq.mmio.write(SEQ_ENABLE, 0)
        raise

print()
print(f"Completed {len(results)}/{LOOP_COUNT} shots.")
all_margins = [r['min_margin_s'] for r in results if r['min_margin_s'] is not None]
if all_margins:
    print(f"Worst (smallest) refill margin seen across all shots: {min(all_margins)*1e3:+.4f} ms")
    if min(all_margins) < 0:
        print(
            "FAIL: at least one refill transfer() call completed "
            "after the next consuming row fired."
        )
    else:
        print(
            "PASS: every refill transfer() call completed before "
            "its next consuming row fired."
        )


seq.mmio.write(SEQ_TTL_MODE, 0)
seq.mmio.write(SEQ_ENABLE, 0)
print('Done.')


  refill TX_B chunk 1: gap-before-row=2, consume-row=3, issued=42.1100 ms, transfer_call=0.1958 ms, margin=+1.9100 ms
  refill TX_B chunk 2: gap-before-row=4, consume-row=5, issued=64.3504 ms, transfer_call=0.2639 ms, margin=+1.6796 ms
  refill TX_B chunk 3: gap-before-row=6, consume-row=7, issued=86.2736 ms, transfer_call=0.1884 ms, margin=+1.7664 ms
  refill TX_B chunk 4: gap-before-row=8, consume-row=9, issued=108.2409 ms, transfer_call=0.1739 ms, margin=+1.8091 ms
  refill TX_B chunk 5: gap-before-row=10, consume-row=11, issued=130.1475 ms, transfer_call=0.1556 ms, margin=+1.9125 ms
  refill TX_B chunk 6: gap-before-row=12, consume-row=13, issued=152.3674 ms, transfer_call=0.2627 ms, margin=+1.7026 ms
  refill TX_B chunk 7: gap-before-row=14, consume-row=15, issued=174.2465 ms, transfer_call=0.1460 ms, margin=+1.8335 ms
  refill TX_B chunk 8: gap-before-row=16, consume-row=17, issued=196.2323 ms, transfer_call=0.1478 ms, margin=+1.8577 ms
  refill TX_B chunk 9: gap-before-row=18, c